## Collaborative Filtering

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import csr_matrix
from implicit import als

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/collaborative"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Ready!")

Ready!


In [2]:
print("Loading ratings...")
ratings = pd.read_csv(f"{PROCESSED_DIR}/collaborative_ratings.csv")
movies  = pd.read_csv(f"{PROCESSED_DIR}/movies_master.csv")[["movie_id","title","genres","poster_path"]]

print(f"Ratings  : {len(ratings):,}")
print(f"Users    : {ratings['userId'].nunique():,}")
print(f"Movies   : {ratings['movie_id'].nunique():,}")
print(f"Avg rating: {ratings['rating'].mean():.2f}")
ratings.head()

Loading ratings...
Ratings  : 21,451,796
Users    : 156,066
Movies   : 7,292
Avg rating: 3.56


,userId,movie_id,rating
0,1,680,5.0
1,3,680,5.0
2,4,680,4.0
3,5,680,4.0
4,7,680,4.0


In [3]:
# تحويل IDs لـ indices متسلسلة
user_ids  = ratings["userId"].unique()
movie_ids = ratings["movie_id"].unique()

user2idx  = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}
idx2movie = {i: m for m, i in movie2idx.items()}

ratings["user_idx"]  = ratings["userId"].map(user2idx)
ratings["movie_idx"] = ratings["movie_id"].map(movie2idx)

print(f"Users  : {len(user2idx):,}")
print(f"Movies : {len(movie2idx):,}")

# بناء الـ Sparse Matrix (movies × users)
# implicit بيحتاج movie × user مش user × movie
sparse_matrix = csr_matrix(
    (ratings["rating"].values,
     (ratings["movie_idx"].values, ratings["user_idx"].values)),
    shape=(len(movie2idx), len(user2idx))
)

print(f"Matrix shape : {sparse_matrix.shape}")
print(f"Density      : {sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]):.6f}")

Users  : 156,066
Movies : 7,292
Matrix shape : (7292, 156066)
Density      : 0.018850


In [4]:
print("Training ALS model...")
print("(This may take 2-3 minutes)")

model = als.AlternatingLeastSquares(
    factors          = 100,   # حجم الـ embeddings
    iterations       = 30,    # عدد التكرارات
    regularization   = 0.1,
    alpha            = 40,    # confidence weight
    use_gpu          = False,
    calculate_training_loss = True,
    random_state     = 42
)

model.fit(sparse_matrix)
print("Training complete!")

Training ALS model...
(This may take 2-3 minutes)


C:\Users\Saeed\Desktop\neural_network_project\venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 20 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

Training complete!


In [5]:
def recommend_by_movie(movie_title: str, n: int = 10) -> pd.DataFrame:
    """ترشيح أفلام مشابهة بناءً على سلوك المستخدمين"""
    
    title_lower = movie_title.lower()
    match = movies[movies["title"].str.lower().str.contains(title_lower, na=False)]
    
    if match.empty:
        print(f"'{movie_title}' not found!")
        return pd.DataFrame()
    
    movie_id   = match.iloc[0]["movie_id"]
    movie_name = match.iloc[0]["title"]
    
    if movie_id not in movie2idx:
        print(f"'{movie_name}' has no ratings data!")
        return pd.DataFrame()
    
    movie_idx = movie2idx[movie_id]
    
    # طلب أكتر عشان نفلتر اللي مش موجود
    similar = model.similar_items(movie_idx, N=n + 20)
    similar_indices = similar[0]
    similar_scores  = similar[1]
    
    results = []
    for idx, score in zip(similar_indices, similar_scores):
        idx = int(idx)
        # تخطي الفيلم نفسه والـ indices الغريبة
        if idx == movie_idx:
            continue
        if idx not in idx2movie:
            continue
        mid = idx2movie[idx]
        movie_row = movies[movies["movie_id"] == mid]
        if movie_row.empty:
            continue
        results.append({
            "movie_id"         : mid,
            "title"            : movie_row.iloc[0]["title"],
            "genres"           : movie_row.iloc[0]["genres"],
            "similarity_score" : round(float(score), 4)
        })
        if len(results) == n:
            break
    
    print(f"Similar to: {movie_name}")
    return pd.DataFrame(results)

print("recommend_by_movie() ready!")

recommend_by_movie() ready!


In [6]:
def recommend_for_user(user_id: int, n: int = 10) -> pd.DataFrame:
    """ترشيح أفلام لمستخدم معين"""
    
    if user_id not in user2idx:
        print(f"User {user_id} not found!")
        return pd.DataFrame()
    
    user_idx    = user2idx[user_id]
    user_sparse = sparse_matrix.T.tocsr()
    
    # نطلب 500 candidate عشان نضمن نلاقي n بعد الفلترة
    recommended = model.recommend(
        userid                     = user_idx,
        user_items                 = user_sparse[user_idx],
        N                          = 500,
        filter_already_liked_items = True,
        recalculate_user           = True
    )
    
    rec_indices = recommended[0]
    rec_scores  = recommended[1]
    
    # بناء mapping سريع من movie_id لـ row
    movies_indexed = movies.set_index("movie_id")
    
    results = []
    for idx, score in zip(rec_indices, rec_scores):
        idx = int(idx)
        if idx not in idx2movie:
            continue
        mid = idx2movie[idx]
        if mid not in movies_indexed.index:
            continue
        row = movies_indexed.loc[mid]
        results.append({
            "movie_id"             : mid,
            "title"                : row["title"],
            "genres"               : row["genres"],
            "recommendation_score" : round(float(score), 4)
        })
        if len(results) == n:
            break
    
    watched_count = len(ratings[ratings["userId"] == user_id])
    print(f"User {user_id} watched {watched_count} movies")
    
    return pd.DataFrame(results)

print("recommend_for_user() ready!")

recommend_for_user() ready!


In [7]:
test_movies = ["The Dark Knight", "Toy Story", "The Godfather", "Inception"]

print("=== Similar Movies (Collaborative) ===\n")
for movie in test_movies:
    recs = recommend_by_movie(movie, n=5)
    if not recs.empty:
        print(f"\n[{movie}]")
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} score: {row['similarity_score']:.4f}")

=== Similar Movies (Collaborative) ===

Similar to: The Dark Knight

[The Dark Knight]
  → Pirates of the Caribbean: Dead Men Tell No Tales score: 0.6407
Similar to: Toy Story
Similar to: The Godfather
Similar to: Inception

[Inception]
  → Love Comes Softly                        score: 0.7246
  → Superhero Movie                          score: 0.7190


In [8]:
# اختار user عشوائي
sample_user = ratings["userId"].sample(1, random_state=42).values[0]
print(f"Testing with User ID: {sample_user}")

# الأفلام اللي شافها
watched = ratings[ratings["userId"] == sample_user].merge(movies, on="movie_id")
print(f"\nWatched movies (sample):")
print(watched[["title","rating"]].sort_values("rating", ascending=False).head(5).to_string())

print("\nRecommendations:")
recs = recommend_for_user(sample_user, n=10)
print(recs[["title","genres","recommendation_score"]].to_string())

Testing with User ID: 103270

Watched movies (sample):
                       title  rating
0               Pulp Fiction     5.0
91                     Se7en     5.0
233  A River Runs Through It     5.0
85                 Airplane!     5.0
83             Groundhog Day     5.0

Recommendations:
User 103270 watched 334 movies
                          title                        genres  recommendation_score
0                    Supercop 2  Action|Crime|Thriller|Comedy                0.7445
1                       Sicario         Action|Crime|Thriller                0.6912
2                  Nevada Smith                       Western                0.6889
3  The Killing of a Sacred Deer        Drama|Thriller|Mystery                0.6838
4              American Wedding                Comedy|Romance                0.6744
5                           Max        Adventure|Drama|Family                0.6662
6                   Han Gong-ju                         Drama                0.6644
7 

In [9]:
print("Saving model...")

joblib.dump(model,      f"{MODELS_DIR}/als_model.pkl")
joblib.dump(user2idx,   f"{MODELS_DIR}/user2idx.pkl")
joblib.dump(movie2idx,  f"{MODELS_DIR}/movie2idx.pkl")
joblib.dump(idx2movie,  f"{MODELS_DIR}/idx2movie.pkl")

from scipy.sparse import save_npz
save_npz(f"{MODELS_DIR}/sparse_matrix.npz", sparse_matrix)

print("Saved:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024 / 1024
    print(f"  {f:30s} {size:.1f} MB")

Saving model...
Saved:
  als_model.pkl                  63.0 MB
  idx2movie.pkl                  0.2 MB
  movie2idx.pkl                  0.2 MB
  sparse_matrix.npz              45.9 MB
  user2idx.pkl                   3.4 MB
